In [6]:
import pandas as pd
import psycopg2
import os

conn = psycopg2.connect(
    host = 'localhost',
    port = '5432',
    database = 'postgres',
    user = 'postgres',
    password = ''
)

cur = conn.cursor()


query_create = """
-- 务必加分号
DROP TABLE IF EXISTS hot_drink_retail.final_simulation_base;

CREATE TABLE hot_drink_retail.final_simulation_base AS
SELECT
    s.name,
    s.location_type,
    t.*
FROM hot_drink_retail.beijing_drink_shops_processed AS s
CROSS JOIN  hot_drink_retail.temperature_trend_merged_data AS t;
"""


cur.execute(query_create)
conn.commit()


df_sql = pd.read_sql_query('SELECT * FROM hot_drink_retail.final_simulation_base LIMIT 10', conn)

cur.close()
conn.close()

# 展示结果
display(df_sql.head())

C:\Users\Administrator\AppData\Local\Temp\ipykernel_5908\1304572318.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_sql = pd.read_sql_query('SELECT * FROM hot_drink_retail.final_simulation_base LIMIT 10', conn)


,name,location_type,date,coffee_index,milktea_index,tavg,tmin,tmax
0,喜茶(北京王府井喜悦购物中心店),抗寒：重度商务刚需,2025/2/28,592,524,9.6,-2.1,19.1
1,喜茶(北京王府井喜悦购物中心店),抗寒：重度商务刚需,2025/3/1,446,768,9.7,2.1,19.1
2,喜茶(北京王府井喜悦购物中心店),抗寒：重度商务刚需,2025/3/2,443,740,2.7,-1.9,15.6
3,喜茶(北京王府井喜悦购物中心店),抗寒：重度商务刚需,2025/3/3,557,557,5.0,-3.2,9.2
4,喜茶(北京王府井喜悦购物中心店),抗寒：重度商务刚需,2025/3/4,589,593,4.4,1.8,10.1


In [8]:
import pandas as pd
import psycopg2


conn = psycopg2.connect(database="postgres", user="postgres", host="localhost", port="5432")

df_shops = pd.read_sql("SELECT name, location_type FROM hot_drink_retail.beijing_drink_shops_processed", conn)
df_trends = pd.read_sql("SELECT * FROM hot_drink_retail.temperature_trend_merged_data", conn)



df_shops['key'] = 1
df_trends['key'] = 1


df_master_pd = pd.merge(df_shops, df_trends, on='key').drop('key', axis=1)

print(f"门店数: {len(df_shops)} | 天数: {len(df_trends)} | 总行数: {len(df_master_pd)}")

# 4. 查看结果
display(df_master_pd.head())

conn.close()

门店数: 557 | 天数: 364 | 总行数: 202748


C:\Users\Administrator\AppData\Local\Temp\ipykernel_5908\2705840122.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_shops = pd.read_sql("SELECT name, location_type FROM hot_drink_retail.beijing_drink_shops_processed", conn)
C:\Users\Administrator\AppData\Local\Temp\ipykernel_5908\2705840122.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_trends = pd.read_sql("SELECT * FROM hot_drink_retail.temperature_trend_merged_data", conn)


,name,location_type,date,coffee_index,milktea_index,tavg,tmin,tmax
0,喜茶(北京王府井喜悦购物中心店),抗寒：重度商务刚需,2025/2/28,592,524,9.6,-2.1,19.1
1,喜茶(北京王府井喜悦购物中心店),抗寒：重度商务刚需,2025/3/1,446,768,9.7,2.1,19.1
2,喜茶(北京王府井喜悦购物中心店),抗寒：重度商务刚需,2025/3/2,443,740,2.7,-1.9,15.6
3,喜茶(北京王府井喜悦购物中心店),抗寒：重度商务刚需,2025/3/3,557,557,5.0,-3.2,9.2
4,喜茶(北京王府井喜悦购物中心店),抗寒：重度商务刚需,2025/3/4,589,593,4.4,1.8,10.1


In [9]:
import numpy as np

def get_sales_prediction(row):
    """业务逻辑：将气压、店型和热度趋势转化为一个销售数字"""
    base_sales = 100
    avg_index = (row['milktea_index'] + row['coffee_index']) / 2.0
    trend_factor = avg_index / 50.0 

    temp = row['tmin']
    location_type = row['location_type']
    
    # 默认不影响
    weather_impact = 1.0
    
    # 【核心逻辑】：针对不同店型设置不同的耐寒系数
    if temp < 2:
        if location_type == '脆弱：极度依赖天气':
            weather_impact = max(0.2, 1.0 + (temp * 0.05)) # 景区最怕冷
        elif location_type == '抗寒：重度商务刚需':
            weather_impact = max(0.9, 1.0 + (temp * 0.005)) # 职场店最硬
        elif location_type == '中庸：混合社区':
            weather_impact = max(0.2, 1.0 + (temp * 0.02)) # 社区店适中
        else:
            weather_impact = 1.0
            
    # 计算最终值 + 随机震荡(噪音)
    prediction = (base_sales * trend_factor * weather_impact) + np.random.normal(0, 5)
    return round(prediction, 1)

In [11]:
df_master_pd['pre_sale'] = df_master_pd.apply(get_sales_prediction,axis=1)
display(df_master_pd.sort_values(by='tmin').head(10))

,name,location_type,date,coffee_index,milktea_index,tavg,tmin,tmax,pre_sale
142651,星巴克咖啡(北投新奥购物中心店),中庸：混合社区,2026/1/21,566,583,-9.2,-16.9,0.1,762.0
16343,luckin coffee 瑞幸咖啡(东华门大街店),中庸：混合社区,2026/1/21,566,583,-9.2,-16.9,0.1,766.0
179051,星巴克咖啡(乐成中心SPACE3店),中庸：混合社区,2026/1/21,566,583,-9.2,-16.9,0.1,760.8
98243,Liberté小篱笆,中庸：混合社区,2026/1/21,566,583,-9.2,-16.9,0.1,756.1
178687,星巴克臻选(龙湖北京丽泽天街店),中庸：混合社区,2026/1/21,566,583,-9.2,-16.9,0.1,755.8
16707,蜜雪冰城(前门大街店),中庸：混合社区,2026/1/21,566,583,-9.2,-16.9,0.1,762.9
72035,喜茶(北京鼓楼店),中庸：混合社区,2026/1/21,566,583,-9.2,-16.9,0.1,764.7
178323,瑞幸咖啡(花乡奥莱村店),中庸：混合社区,2026/1/21,566,583,-9.2,-16.9,0.1,761.1
98607,Grid Coffee(前门大街店),中庸：混合社区,2026/1/21,566,583,-9.2,-16.9,0.1,772.0
17071,茶百道(北京王府井银泰in88店),抗寒：重度商务刚需,2026/1/21,566,583,-9.2,-16.9,0.1,1046.3
